In [2]:
!pip install -q transformers torch accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [3]:
!hf auth login

User is already logged in. Use `hf auth login --force` to force re-login.


In [4]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers import BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Initializing {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
)

Initializing meta-llama/Llama-3.2-3B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [5]:
if hasattr(model.model, "_forward_hooks"):
    model.model._forward_hooks.clear()

for layer in model.model.layers:
    if hasattr(layer, "_forward_hooks"):
        layer._forward_hooks.clear()

print("all stale hooks flushed successfully! VRAM is clean.")

def sparsify_activations(hidden_states, keep_ratio=1.0):

    magnitudes = torch.abs(hidden_states)

    channel_dim = hidden_states.size(-1)
    k = int(channel_dim * keep_ratio)

    topk_values, _ = torch.topk(magnitudes, k, dim=-1)

    thresholds = topk_values[..., -1].unsqueeze(-1)

    mask = magnitudes >= thresholds

    return hidden_states * mask
def calculate_topk_entropy(logits, k=10):
    last_token_logits = logits[0, -1, :]

    topk_logits, _ = torch.topk(last_token_logits, k, dim=-1)

    probs = torch.softmax(topk_logits, dim=-1)

    entropy = -torch.sum(probs * torch.log2(probs + 1e-9), dim=-1)

    return entropy.item()
def make_hook(layer_idx):

    def hook(module, input, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
            processed_states = sparsify_activations(hidden_states, keep_ratio=0.35)
            return (processed_states,) + output[1:]

        return sparsify_activations(output, keep_ratio=0.35)
    return hook
num_layers = len(model.model.layers)
start_layer = num_layers // 4
end_layer = (3 * num_layers) // 4

print(f"injecting Sparsification Hooks into layers {start_layer} through {end_layer}...")
for i in range(start_layer, end_layer):
    model.model.layers[i].register_forward_hook(make_hook(i))


all stale hooks flushed successfully! VRAM is clean.
injecting Sparsification Hooks into layers 7 through 21...


In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)




prompt = "whats your opinion on the future of decentralized computing architecture"
messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

print(f"Prompt: \"{prompt}\"\n" + "-" * 60)

input_ids = inputs["input_ids"]
attention_mask = inputs.get("attention_mask", None)

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1000,        # Give the 3B brain real room to answer!
        do_sample=True,

        # The Optimized Llama-3 Sampler Stack
        temperature=0.7,           # Lower temperature for much tighter logical focus
        top_p=0.9,                 # Restrict generation to the top 90% probability mass
        repetition_penalty=1.05    # A tiny, gentle nudge instead of a sledgehammer
    )


generated_text = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
print(f"Engine Output:\n{generated_text}")
print("-" * 60)




Prompt: "whats your opinion on the future of decentralized computing architecture"
------------------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Engine Output:
The decentralized computing architecture is a rapidly evolving field that has the potential to transform the way we approach technology. Here are some key insights:

**Key Benefits**

Decentral computing enables seamless communication, collaboration, and innovation across various industries and organizations.

* **Decentralized networks**: These networks enable collaboration, data exchange, and decentralized systems to provide scalable infrastructure.
* **AI-driven systems**: Decentral AI systems will facilitate efficient decision-making, optimization, and control.

Decentral computing can also facilitate:

1. Data processing: Artificial intelligence and deep learning technologies integrate to optimize computing and improve systems.
2. Cloud computing: Scalability and access to large-scale infrastructure, enabling cost-effective collaboration.

Key aspects include:

* Decentralization: decentralized networks and blockchain-based applications can be integrated with cloud 

In [9]:
import os
import torch
import grpc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import sys
!{sys.executable} -m pip install --upgrade grpcio
import warnings
warnings.filterwarnings("ignore")

WSL2_PINGGY_ADDRESS = "groac-45-112-149-25.run.pinggy-free.link:38137"

print(f" Establishing Remote gRPC Link to WSL2 Data Plane at: {WSL2_PINGGY_ADDRESS}")
channel = grpc.insecure_channel(WSL2_PINGGY_ADDRESS)

with open("activation_stream.proto", "w") as f:
    f.write("""
    syntax = "proto3";
    package dllm.network;
    message ActivationPayload {
        uint64 prompt_hash = 1;
        uint32 task_serial_number = 2;
        uint32 layer_index = 3;
        float global_min = 4;
        float global_max = 5;
        float scale_factor = 6;
        bytes quantized_tensor = 7;
        bytes sparse_bitmask = 8;
    }
    message StreamAck {
        bool success = 1;
        string error_message = 2;
    }
    service ActivationStream {
        rpc StreamLayerActivations (ActivationPayload) returns (StreamAck);
    }
    """)

os.system("python -m grpc_tools.protoc -I. --python_out=. --grpc_python_out=. activation_stream.proto")

import activation_stream_pb2 as pb2
import activation_stream_pb2_grpc as pb2_grpc
grpc_client = pb2_grpc.ActivationStreamStub(channel)

def stream_tensor_to_go_node(hidden_states, layer_idx, prompt_hash=99999, task_serial=2):
    """Intercepts activations on Colab's T4 GPU and exports them to local WSL2 Go node"""
    magnitudes = torch.abs(hidden_states)
    channel_dim = hidden_states.size(-1)
    k = int(channel_dim * 0.35)
    topk_values, _ = torch.topk(magnitudes, k, dim=-1)
    thresholds = topk_values[..., -1].unsqueeze(-1)
    mask = magnitudes >= thresholds
    sparse_states = hidden_states * mask

    raw_tensor_bytes = sparse_states.detach().cpu().to(torch.float16).numpy().tobytes()
    mock_bitmask = bytes([0b11001100] * 384)

    payload = pb2.ActivationPayload(
        prompt_hash=prompt_hash,
        task_serial_number=task_serial,
        layer_index=layer_idx,
        global_min=-1.0,
        global_max=1.0,
        scale_factor=1.0,
        quantized_tensor=raw_tensor_bytes,
        sparse_bitmask=mock_bitmask
    )
    try:
        response = grpc_client.StreamLayerActivations(payload)
        return response.success
    except Exception as e:
        return False

def make_network_hook(layer_idx):
    def hook(module, input, output):
        if isinstance(output, tuple):
            stream_tensor_to_go_node(output[0], layer_idx)
            return output
        stream_tensor_to_go_node(output, layer_idx)
        return output
    return hook

def main():

    if hasattr(model.model, "_forward_hooks"): model.model._forward_hooks.clear()
    for layer in model.model.layers:
        if hasattr(layer, "_forward_hooks"): layer._forward_hooks.clear()

    print("Injecting Remote-Slicing Data Plane Hooks into Layers 6 through 12...")
    for i in range(6, 12):
        model.model.layers[i].register_forward_hook(make_network_hook(i))

    prompt = "whats your opinion on the future of decentralized computing architecture in short"
    messages = [{"role": "user", "content": prompt}]
    inputs = inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)["input_ids"]

    print("\nFiring Cloud GPU Engine Pass -> Streaming Activations to Local WSL2...")
    with torch.no_grad():
        model.generate(inputs, max_new_tokens=5, do_sample=False)
    print("🏁 Remote cloud processing complete.")

if __name__ == "__main__":
    main()


🔗 Establishing Remote gRPC Link to WSL2 Data Plane at: groac-45-112-149-25.run.pinggy-free.link:38137
⚡ Injecting Remote-Slicing Data Plane Hooks into Layers 6 through 12...

🚀 Firing Cloud GPU Engine Pass -> Streaming Activations to Local WSL2...
🏁 Remote cloud processing complete.


In [1]:
!pip install grpcio grpcio-tools

In [13]:
!pip install --upgrade grpcio